# Qwen2.5-7B-Instruct QLoRA Fine-Tuning v2 (FitMyResume)


## 0. Sanity check the runtime

In [1]:
!nvidia-smi

Thu Jun  4 17:58:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Mount Drive and set paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

DRIVE_BASE = '/content/drive/MyDrive/fit-my-resume'
SFT_TRAIN_PATH = f'{DRIVE_BASE}/data/instruction_tuning_train.jsonl'
SFT_VAL_PATH = f'{DRIVE_BASE}/data/instruction_tuning_validation.jsonl'

OUTPUT_DIR = f'{DRIVE_BASE}/models/qwen25-7b-fitmyresume-lora-v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(SFT_TRAIN_PATH), f'Missing: {SFT_TRAIN_PATH}'
assert os.path.exists(SFT_VAL_PATH), f'Missing: {SFT_VAL_PATH}'
print('SFT files found.')
print(f'  train:  {SFT_TRAIN_PATH}')
print(f'  val:    {SFT_VAL_PATH}')
print(f'  output: {OUTPUT_DIR}')

SFT files found.
  train:  /content/drive/MyDrive/fit-my-resume/data/instruction_tuning_train.jsonl
  val:    /content/drive/MyDrive/fit-my-resume/data/instruction_tuning_validation.jsonl
  output: /content/drive/MyDrive/fit-my-resume/models/qwen25-7b-fitmyresume-lora-v2


## 2. Install dependencies

Note: `trl>=0.13.0` is required for `assistant_only_loss`. Other versions match v1.

In [4]:
!pip install -q -U \
    'bitsandbytes>=0.45.0' \
    'transformers>=4.46.0' \
    'peft>=0.14.0' \
    'trl>=0.13.0' \
    'accelerate>=1.1.0' \
    'datasets>=3.1.0' \
    'sentencepiece' \
    'protobuf'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.35.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
google-cloud-aiplatform 1.153.1 requires protobuf!=4.21.0

In [5]:
import torch, trl, transformers, peft
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('bf16 supported:', torch.cuda.is_bf16_supported())
print(f'trl:          {trl.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'peft:         {peft.__version__}')

# Confirm assistant_only_loss is supported in this trl version
from trl import SFTConfig
import inspect
sig = inspect.signature(SFTConfig.__init__)
assert 'assistant_only_loss' in sig.parameters, (
    'Your trl version is too old for assistant_only_loss. Upgrade trl and restart runtime.'
)
print('assistant_only_loss is supported. ✓')

CUDA available: True
Device: NVIDIA A100-SXM4-40GB
bf16 supported: True
trl:          1.5.1
transformers: 5.10.1
peft:         0.19.1
assistant_only_loss is supported. ✓


**If the assertion above fails:** Restart the runtime (Runtime → Restart session) and re-run all cells from the top. pip installed new versions but Python kept the old ones in memory.

## 3. Load Qwen2.5-7B-Instruct in 4-bit quantization

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    dtype=torch.bfloat16,  # 'dtype' replaced the deprecated 'torch_dtype'
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f'Model loaded. Vocab size: {len(tokenizer)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded. Vocab size: 151665


## 4. Apply LoRA adapter (rank 32 — v2 change)


In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


## 5. Load and format the dataset

In [8]:
from datasets import load_dataset

raw_train = load_dataset('json', data_files=SFT_TRAIN_PATH, split='train')
raw_val = load_dataset('json', data_files=SFT_VAL_PATH, split='train')

print(f'train rows: {len(raw_train)}')
print(f'val rows:   {len(raw_val)}')
print(f'\nSample row keys: {list(raw_train[0].keys())}')

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

train rows: 5855
val rows:   726

Sample row keys: ['instruction', 'input', 'output', 'metadata']


### Format using Qwen's chat template


In [9]:
def to_messages(example):
    """
    Return a messages list. SFTTrainer applies the chat template internally
    and uses the role boundaries to mask non-assistant tokens from the loss.
    """
    return {
        'messages': [
            {'role': 'system', 'content': example['instruction']},
            {'role': 'user', 'content': example['input']},
            {'role': 'assistant', 'content': example['output']},
        ]
    }

train_ds = raw_train.map(to_messages, remove_columns=raw_train.column_names)
val_ds = raw_val.map(to_messages, remove_columns=raw_val.column_names)

print('Sample messages structure:')
for msg in train_ds[0]['messages']:
    content_preview = msg['content'][:120].replace('\n', ' ')
    print(f'  [{msg["role"]:9s}] {content_preview}...')

Map:   0%|          | 0/5855 [00:00<?, ? examples/s]

Map:   0%|          | 0/726 [00:00<?, ? examples/s]

Sample messages structure:
  [system   ] Evaluate the resume against the job description. Return only valid JSON with score, explanation, and resume_suggestions....
  [user     ] RESUME: CONSULTANT Summary I am an experienced Program Manager, delivering enterprise-grade on-premises and SaaS product...
  [assistant] {"score":55,"explanation":{"matched_qualifications":["Extensive experience with Windows OS, Active Directory, and Group ...


In [10]:
# Token length sanity check with the new 6144 ceiling
import statistics

def total_tokens(messages):
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return len(tokenizer.encode(rendered))

lengths = [total_tokens(ex['messages']) for ex in train_ds.select(range(200))]
print(f'Sample of 200 train rows:')
print(f'  min:    {min(lengths)}')
print(f'  max:    {max(lengths)}')
print(f'  mean:   {statistics.mean(lengths):.0f}')
print(f'  median: {statistics.median(lengths):.0f}')
print(f'  >4096:  {sum(1 for x in lengths if x > 4096)}/200 ({100*sum(1 for x in lengths if x > 4096)/200:.1f}%)')
print(f'  >6144:  {sum(1 for x in lengths if x > 6144)}/200 ({100*sum(1 for x in lengths if x > 6144)/200:.1f}%)')

Sample of 200 train rows:
  min:    660
  max:    5197
  mean:   2311
  median: 2224
  >4096:  5/200 (2.5%)
  >6144:  0/200 (0.0%)


## 6. Configure SFTTrainer

In [11]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=3e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.0,
    optim='paged_adamw_8bit',
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    max_length=6144,
    packing=False,
    assistant_only_loss=True,
    logging_steps=10,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    seed=42,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=sft_config,
)

# Step count estimate
steps_per_epoch = len(train_ds) // (sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)
total_steps = steps_per_epoch * sft_config.num_train_epochs
print(f'Steps per epoch: {steps_per_epoch}')
print(f'Total steps:     {total_steps}')
print(f'Rough A100 estimate: ~{total_steps * 50 / 60:.0f} min ({total_steps * 50 / 3600:.1f} hrs)')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/5855 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/726 [00:00<?, ? examples/s]

Steps per epoch: 365
Total steps:     730
Rough A100 estimate: ~608 min (10.1 hrs)


## 7. Full training


In [12]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,0.682252,0.679172,0.663586,7295579.000000,0.791199
400,0.555115,0.655293,0.558882,14674866.000000,0.797168
600,0.539001,0.636983,0.543009,22012963.000000,0.802127
732,0.530171,0.634900,0.543925,26860592.000000,0.802807


TrainOutput(global_step=732, training_loss=0.6315687712424439, metrics={'train_runtime': 13440.2761, 'train_samples_per_second': 0.871, 'train_steps_per_second': 0.054, 'total_flos': 1.1525384967149814e+18, 'train_loss': 0.6315687712424439, 'epoch': 2.0})

## 8. Save the final adapter

Saves to `models/qwen25-7b-fitmyresume-lora-v2/final/`.

In [13]:
FINAL_ADAPTER_PATH = f'{OUTPUT_DIR}/final'
trainer.model.save_pretrained(FINAL_ADAPTER_PATH)
tokenizer.save_pretrained(FINAL_ADAPTER_PATH)
print(f'Adapter saved to: {FINAL_ADAPTER_PATH}')
!ls -lh "{FINAL_ADAPTER_PATH}"

Adapter saved to: /content/drive/MyDrive/fit-my-resume/models/qwen25-7b-fitmyresume-lora-v2/final
total 319M
-rw------- 1 root root 1.1K Jun  4 21:45 adapter_config.json
-rw------- 1 root root 309M Jun  4 21:45 adapter_model.safetensors
-rw------- 1 root root 2.5K Jun  4 21:45 chat_template.jinja
-rw------- 1 root root 5.1K Jun  4 21:45 README.md
-rw------- 1 root root  694 Jun  4 21:45 tokenizer_config.json
-rw------- 1 root root  11M Jun  4 21:45 tokenizer.json


In [14]:
# Save the training history for the report
import json
history_path = f'{OUTPUT_DIR}/training_history.json'
with open(history_path, 'w') as f:
    json.dump(trainer.state.log_history, f, indent=2)
print(f'Training history saved to: {history_path}')

# Print the final eval metrics for quick reference
eval_entries = [e for e in trainer.state.log_history if 'eval_loss' in e]
print(f'\nEval checkpoints during training ({len(eval_entries)} total):')
print(f'{"step":>6} {"train_loss":>12} {"eval_loss":>10}')
for e in eval_entries:
    step = e.get('step', '?')
    train_loss = e.get('train_loss', '?')
    eval_loss = e.get('eval_loss', '?')
    print(f'{step:>6} {str(train_loss):>12} {eval_loss:>10.4f}')

Training history saved to: /content/drive/MyDrive/fit-my-resume/models/qwen25-7b-fitmyresume-lora-v2/training_history.json

Eval checkpoints during training (4 total):
  step   train_loss  eval_loss
   200            ?     0.6792
   400            ?     0.6553
   600            ?     0.6370
   732            ?     0.6349


## 9. Quick inference sanity check

Run the v2 model on one validation example. Compare to the teacher's output.

In [15]:
import json

sample_idx = 0
sample = raw_val[sample_idx]
print(f'=== Validation sample [{sample_idx}] ===')
print(f'pair_id: {sample["metadata"]["pair_id"]}')
print(f'strategy: {sample["metadata"]["pairing_strategy"]}')
print(f'\n--- TEACHER OUTPUT (first 600 chars) ---')
print(sample['output'][:600])

=== Validation sample [0] ===
pair_id: validation_10149490_job_000176_strong_hybrid
strategy: strong_hybrid

--- TEACHER OUTPUT (first 600 chars) ---
{"score":35,"explanation":{"matched_qualifications":["Managed construction schedules, manpower loading, and resource loading for refinery projects (e.g., VIP Project with 500 construction employees).","Oversaw project budgets and cost control, including budget reviews and forecasts.","Coordinated with multiple internal teams (Operations, Project Engineering) and external contractors.","18 years of supervisory experience managing large construction crews and multiple projects simultaneously."],"missing_or_weak_qualifications":["Bachelor's degree in construction, project management, or related f


In [16]:
messages = [
    {'role': 'system', 'content': sample['instruction']},
    {'role': 'user', 'content': sample['input']},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

model.config.use_cache = True
model.eval()

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
    )

generated = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== STUDENT v2 OUTPUT ===')
print(generated)

try:
    parsed = json.loads(generated)
    print('\n✓ Parsed as valid JSON')
    print(f'  score: {parsed.get("score")} (teacher: {json.loads(sample["output"]).get("score")})')
    print(f'  matched:    {len(parsed.get("explanation", {}).get("matched_qualifications", []))}')
    print(f'  missing:    {len(parsed.get("explanation", {}).get("missing_or_weak_qualifications", []))}')
    print(f'  suggestions: {len(parsed.get("resume_suggestions", []))}')
except json.JSONDecodeError as e:
    print(f'\n✗ JSON parse failed: {e}')

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


=== STUDENT v2 OUTPUT ===
{"score":35,"explanation":{"matched_qualifications":["Managed construction projects including OSBL, ISBL, and FGS projects, demonstrating ability to oversee construction processes.","Experience with budget forecasts, cost control, and resource allocation, aligning with financial oversight responsibilities.","Supervised large crews (up to 500 employees) and managed multiple projects simultaneously, showing supervisory capability.","Led safety programs and meetings, indicating ability to coordinate with teams and ensure compliance."],"missing_or_weak_qualifications":["No experience in fast-food or restaurant construction projects; background is in industrial/refinery settings.","Lack of direct experience with vendor invoice management, GC payment tracking, or project budget ownership.","No mention of turnover process from construction to operations or training of staff.","Missing bachelor's degree in construction project management, engineering, or related field

## 10. Done
**The v2 adapter is at:**
```
/content/drive/MyDrive/fit-my-resume/models/qwen25-7b-fitmyresume-lora-v2/final
```
